In [4]:
import sys
import os
import json
import torch
from safetensors.torch import load_file

# Add the root folder to sys.path
sys.path.append('/home/vanshnawander/accelerated-hpc/mlsys26-contest')

# Load the JSONL configuration file
jsonl_path = '/home/vanshnawander/accelerated-hpc/mlsys26-contest/workloads/moe/moe_fp8_block_scale_ds_routing_topk8_ng8_kg4_e32_h7168_i2048.jsonl'

workloads = []
with open(jsonl_path, 'r') as f:
    for line in f:
        workloads.append(json.loads(line))

print(f"Loaded {len(workloads)} workload configurations")
print(f"Base directory: {os.path.dirname(jsonl_path)}")

Loaded 19 workload configurations
Base directory: /home/vanshnawander/accelerated-hpc/mlsys26-contest/workloads/moe


In [5]:
# Extract and analyze all workload configurations
for i, workload in enumerate(workloads):
    print(f"\n=== Workload {i+1} ===")
    print(f"UUID: {workload['workload']['uuid']}")
    print(f"Sequence Length: {workload['workload']['axes']['seq_len']}")
    print(f"Local Expert Offset: {workload['workload']['inputs']['local_expert_offset']['value']}")
    print(f"Routed Scaling Factor: {workload['workload']['inputs']['routed_scaling_factor']['value']}")
    
    # Extract file paths
    routing_logits_path = workload['workload']['inputs']['routing_logits']['path']
    routing_bias_path = workload['workload']['inputs']['routing_bias']['path']
    
    print(f"Routing Logits File: {os.path.basename(routing_logits_path)}")
    print(f"Routing Bias File: {os.path.basename(routing_bias_path)}")


=== Workload 1 ===
UUID: b8f4f012-a32e-4356-b4e1-7665b3d598af
Sequence Length: 7
Local Expert Offset: 192
Routed Scaling Factor: 2.5
Routing Logits File: moe_fp8_block_scale_ds_routing_topk8_ng8_kg4_e32_h7168_i2048_547d24f37f554e2fab107fb57a41e73e.safetensors
Routing Bias File: moe_fp8_block_scale_ds_routing_topk8_ng8_kg4_e32_h7168_i2048_547d24f37f554e2fab107fb57a41e73e.safetensors

=== Workload 2 ===
UUID: e05c6c03-5603-4a1c-b34c-dcce0ecaeea4
Sequence Length: 1
Local Expert Offset: 32
Routed Scaling Factor: 2.5
Routing Logits File: moe_fp8_block_scale_ds_routing_topk8_ng8_kg4_e32_h7168_i2048_25ff432053b5474d86dda63b7daf1734.safetensors
Routing Bias File: moe_fp8_block_scale_ds_routing_topk8_ng8_kg4_e32_h7168_i2048_25ff432053b5474d86dda63b7daf1734.safetensors

=== Workload 3 ===
UUID: 6230e838-67ca-41dd-a9d6-6f36b7676c6b
Sequence Length: 32
Local Expert Offset: 32
Routed Scaling Factor: 2.5
Routing Logits File: moe_fp8_block_scale_ds_routing_topk8_ng8_kg4_e32_h7168_i2048_963f2f970f2e4

In [6]:
# Load and analyze tensor dimensions for kernel development
base_dir = '/home/vanshnawander/accelerated-hpc/mlsys26-contest/blob/workloads/moe/moe_fp8_block_scale_ds_routing_topk8_ng8_kg4_e32_h7168_i2048'

tensor_analysis = []

for i, workload in enumerate(workloads):
    seq_len = workload['workload']['axes']['seq_len']
    expert_offset = workload['workload']['inputs']['local_expert_offset']['value']
    
    # Get file paths
    logits_file = os.path.basename(workload['workload']['inputs']['routing_logits']['path'])
    bias_file = os.path.basename(workload['workload']['inputs']['routing_bias']['path'])
    
    # Load tensors
    logits_path = os.path.join(base_dir, logits_file)
    bias_path = os.path.join(base_dir, bias_file)
    
    logits_data = load_file(logits_path)
    bias_data = load_file(bias_path)
    
    routing_logits = logits_data['routing_logits']
    routing_bias = bias_data['routing_bias']
    
    analysis = {
        'workload_id': i+1,
        'uuid': workload['workload']['uuid'],
        'seq_len': seq_len,
        'expert_offset': expert_offset,
        'logits_shape': routing_logits.shape,
        'bias_shape': routing_bias.shape,
        'logits_file': logits_file,
        'bias_file': bias_file
    }
    
    tensor_analysis.append(analysis)
    
    print(f"\nWorkload {i+1} - Tensor Analysis:")
    print(f"  Sequence Length: {seq_len}")
    print(f"  Expert Offset: {expert_offset}")
    print(f"  Routing Logits Shape: {routing_logits.shape}")
    print(f"  Routing Bias Shape: {routing_bias.shape}")
    print(f"  Logits Dimensions: [seq_len={routing_logits.shape[0]}, num_experts={routing_logits.shape[1]}]")
    print(f"  Bias Dimensions: [num_experts={routing_bias.shape[0]}]")


Workload 1 - Tensor Analysis:
  Sequence Length: 7
  Expert Offset: 192
  Routing Logits Shape: torch.Size([7, 256])
  Routing Bias Shape: torch.Size([256])
  Logits Dimensions: [seq_len=7, num_experts=256]
  Bias Dimensions: [num_experts=256]

Workload 2 - Tensor Analysis:
  Sequence Length: 1
  Expert Offset: 32
  Routing Logits Shape: torch.Size([1, 256])
  Routing Bias Shape: torch.Size([256])
  Logits Dimensions: [seq_len=1, num_experts=256]
  Bias Dimensions: [num_experts=256]

Workload 3 - Tensor Analysis:
  Sequence Length: 32
  Expert Offset: 32
  Routing Logits Shape: torch.Size([32, 256])
  Routing Bias Shape: torch.Size([256])
  Logits Dimensions: [seq_len=32, num_experts=256]
  Bias Dimensions: [num_experts=256]

Workload 4 - Tensor Analysis:
  Sequence Length: 80
  Expert Offset: 96
  Routing Logits Shape: torch.Size([80, 256])
  Routing Bias Shape: torch.Size([256])
  Logits Dimensions: [seq_len=80, num_experts=256]
  Bias Dimensions: [num_experts=256]

Workload 5 - Ten

In [7]:
# Kernel Development Parameters Analysis
print("=== KERNEL DEVELOPMENT PARAMETERS ===")
print()

# Extract key parameters for kernel design
seq_lengths = [w['seq_len'] for w in tensor_analysis]
expert_offsets = [w['expert_offset'] for w in tensor_analysis]
logits_shapes = [w['logits_shape'] for w in tensor_analysis]

print("SEQUENCE LENGTHS:")
print(f"  Range: {min(seq_lengths)} to {max(seq_lengths)}")
print(f"  Unique values: {sorted(set(seq_lengths))}")
print(f"  Total workloads: {len(seq_lengths)}")

print("\nEXPERT OFFSETS:")
print(f"  Range: {min(expert_offsets)} to {max(expert_offsets)}")
print(f"  Unique values: {sorted(set(expert_offsets))}")

print("\nROUTING LOGITS SHAPES:")
unique_shapes = set(logits_shapes)
for shape in sorted(unique_shapes):
    count = logits_shapes.count(shape)
    print(f"  {shape}: {count} workloads")

print("\nFIXED PARAMETERS:")
print(f"  Number of Experts (routing_logits dim 1): {tensor_analysis[0]['logits_shape'][1]}")
print(f"  Bias Size (routing_bias dim 0): {tensor_analysis[0]['bias_shape'][0]}")
print(f"  Top-K: 8 (from filename)")
print(f"  Num Groups: 8 (from filename)")
print(f"  Experts per Group: 4 (from filename)")

print("\nKERNEL DESIGN IMPLICATIONS:")
print(f"  - Max sequence length to handle: {max(seq_lengths)}")
print(f"  - Expert routing matrix size: [{max(seq_lengths)}, {tensor_analysis[0]['logits_shape'][1]}]")
print(f"  - Bias vector size: [{tensor_analysis[0]['bias_shape'][0]}]")
print(f"  - Need to handle variable sequence lengths efficiently")

=== KERNEL DEVELOPMENT PARAMETERS ===

SEQUENCE LENGTHS:
  Range: 1 to 14107
  Unique values: [1, 7, 14, 15, 16, 32, 52, 53, 54, 55, 56, 57, 58, 59, 62, 80, 901, 11948, 14107]
  Total workloads: 19

EXPERT OFFSETS:
  Range: 0 to 224
  Unique values: [0, 32, 64, 96, 128, 160, 192, 224]

ROUTING LOGITS SHAPES:
  torch.Size([1, 256]): 1 workloads
  torch.Size([7, 256]): 1 workloads
  torch.Size([14, 256]): 1 workloads
  torch.Size([15, 256]): 1 workloads
  torch.Size([16, 256]): 1 workloads
  torch.Size([32, 256]): 1 workloads
  torch.Size([52, 256]): 1 workloads
  torch.Size([53, 256]): 1 workloads
  torch.Size([54, 256]): 1 workloads
  torch.Size([55, 256]): 1 workloads
  torch.Size([56, 256]): 1 workloads
  torch.Size([57, 256]): 1 workloads
  torch.Size([58, 256]): 1 workloads
  torch.Size([59, 256]): 1 workloads
  torch.Size([62, 256]): 1 workloads
  torch.Size([80, 256]): 1 workloads
  torch.Size([901, 256]): 1 workloads
  torch.Size([11948, 256]): 1 workloads
  torch.Size([14107, 2

In [8]:
# Memory and Compute Requirements Analysis
print("=== MEMORY AND COMPUTE ANALYSIS ===")
print()

for analysis in tensor_analysis:
    seq_len = analysis['seq_len']
    num_experts = analysis['logits_shape'][1]
    
    # Memory calculations (assuming bfloat16 = 2 bytes)
    logits_memory_mb = (seq_len * num_experts * 2) / (1024 * 1024)
    bias_memory_mb = (num_experts * 2) / (1024 * 1024)
    total_memory_mb = logits_memory_mb + bias_memory_mb
    
    # Compute requirements for top-k selection
    total_comparisons = seq_len * num_experts
    top_k_operations = seq_len * 8  # top-8 per sequence element
    
    print(f"Workload {analysis['workload_id']} (seq_len={seq_len}):")
    print(f"  Routing Logits Memory: {logits_memory_mb:.3f} MB")
    print(f"  Routing Bias Memory: {bias_memory_mb:.3f} MB")
    print(f"  Total Memory: {total_memory_mb:.3f} MB")
    print(f"  Total Comparisons for Top-K: {total_comparisons:,}")
    print(f"  Top-K Operations: {top_k_operations:,}")
    print()

# Summary for largest workload
max_seq_len = max(seq_lengths)
max_analysis = next(w for w in tensor_analysis if w['seq_len'] == max_seq_len)
num_experts = max_analysis['logits_shape'][1]

print("=== WORST-CASE REQUIREMENTS ===")
print(f"Maximum sequence length: {max_seq_len}")
print(f"Maximum routing logits memory: {(max_seq_len * num_experts * 2) / (1024 * 1024):.3f} MB")
print(f"Maximum comparisons per routing operation: {max_seq_len * num_experts:,}")

=== MEMORY AND COMPUTE ANALYSIS ===

Workload 1 (seq_len=7):
  Routing Logits Memory: 0.003 MB
  Routing Bias Memory: 0.000 MB
  Total Memory: 0.004 MB
  Total Comparisons for Top-K: 1,792
  Top-K Operations: 56

Workload 2 (seq_len=1):
  Routing Logits Memory: 0.000 MB
  Routing Bias Memory: 0.000 MB
  Total Memory: 0.001 MB
  Total Comparisons for Top-K: 256
  Top-K Operations: 8

Workload 3 (seq_len=32):
  Routing Logits Memory: 0.016 MB
  Routing Bias Memory: 0.000 MB
  Total Memory: 0.016 MB
  Total Comparisons for Top-K: 8,192
  Top-K Operations: 256

Workload 4 (seq_len=80):
  Routing Logits Memory: 0.039 MB
  Routing Bias Memory: 0.000 MB
  Total Memory: 0.040 MB
  Total Comparisons for Top-K: 20,480
  Top-K Operations: 640

Workload 5 (seq_len=901):
  Routing Logits Memory: 0.440 MB
  Routing Bias Memory: 0.000 MB
  Total Memory: 0.440 MB
  Total Comparisons for Top-K: 230,656
  Top-K Operations: 7,208

Workload 6 (seq_len=16):
  Routing Logits Memory: 0.008 MB
  Routing Bias 

In [9]:
# Data Type and Precision Analysis
print("=== DATA TYPE AND PRECISION ANALYSIS ===")
print()

# Load a sample file to check data types
sample_logits_path = os.path.join(base_dir, tensor_analysis[0]['logits_file'])
sample_bias_path = os.path.join(base_dir, tensor_analysis[0]['bias_file'])

sample_data = load_file(sample_logits_path)
sample_logits = sample_data['routing_logits']
sample_bias = load_file(sample_bias_path)['routing_bias']

print(f"Routing Logits Data Type: {sample_logits.dtype}")
print(f"Routing Bias Data Type: {sample_bias.dtype}")

# Check if data is quantized
print(f"\nLogits Value Range: [{sample_logits.min().item():.4f}, {sample_logits.max().item():.4f}]")
print(f"Bias Value Range: [{sample_bias.min().item():.4f}, {sample_bias.max().item():.4f}]")

# Check for quantization patterns
unique_logits_values = torch.unique(sample_logits)
unique_bias_values = torch.unique(sample_bias)

print(f"\nUnique Logits Values: {len(unique_logits_values)}")
print(f"Unique Bias Values: {len(unique_bias_values)}")

if len(unique_logits_values) < 1000:
    print("Logits appear to be quantized")
else:
    print("Logits appear to be full precision")

if len(unique_bias_values) < 100:
    print("Bias appear to be quantized")
else:
    print("Bias appear to be full precision")

print(f"\nKERNEL IMPLICATIONS:")
print(f"- Input precision: {sample_logits.dtype}")
print(f"- Need to handle {'quantized' if len(unique_logits_values) < 1000 else 'full-precision'} routing logits")
print(f"- Need to handle {'quantized' if len(unique_bias_values) < 100 else 'full-precision'} routing bias")
print(f"- Bias addition may require dequantization")

=== DATA TYPE AND PRECISION ANALYSIS ===

Routing Logits Data Type: torch.float32
Routing Bias Data Type: torch.bfloat16

Logits Value Range: [-3.9720, 1.3622]
Bias Value Range: [4.9375, 4.9688]

Unique Logits Values: 1792
Unique Bias Values: 2
Logits appear to be full precision
Bias appear to be quantized

KERNEL IMPLICATIONS:
- Input precision: torch.float32
- Need to handle full-precision routing logits
- Need to handle quantized routing bias
- Bias addition may require dequantization


In [10]:
# Kernel Optimization Opportunities
print("=== KERNEL OPTIMIZATION OPPORTUNITIES ===")
print()

# Analyze workload patterns
seq_len_groups = {}
for analysis in tensor_analysis:
    seq_len = analysis['seq_len']
    if seq_len not in seq_len_groups:
        seq_len_groups[seq_len] = []
    seq_len_groups[seq_len].append(analysis)

print("SEQUENCE LENGTH DISTRIBUTION:")
for seq_len, workloads in sorted(seq_len_groups.items()):
    print(f"  {seq_len}: {len(workloads)} workload(s)")

print(f"\nOPTIMIZATION INSIGHTS:")
print(f"1. BATCHING OPPORTUNITIES:")
print(f"   - Multiple workloads with same seq_len can be batched")
print(f"   - Consider dynamic batching for efficiency")

print(f"\n2. MEMORY ACCESS PATTERNS:")
print(f"   - Routing logits: [seq_len, {tensor_analysis[0]['logits_shape'][1]}] (row-major)")
print(f"   - Routing bias: [{tensor_analysis[0]['bias_shape'][0]}] (vector)")
print(f"   - Bias broadcast needed across sequence dimension")

print(f"\n3. COMPUTE PATTERNS:")
print(f"   - Top-{8} selection per sequence element")
print(f"   - Expert indices: 0-{tensor_analysis[0]['logits_shape'][1]-1}")
print(f"   - Expert offset handling: 0-{max(expert_offsets)}")

print(f"\n4. KERNEL DESIGN RECOMMENDATIONS:")
print(f"   - Handle variable seq_len with parameterized kernels")
print(f"   - Optimize for row-major access patterns")
print(f"   - Consider shared memory for bias broadcasting")
print(f"   - Implement efficient top-k reduction")
print(f"   - Support both quantized and dequantized inputs")

=== KERNEL OPTIMIZATION OPPORTUNITIES ===

SEQUENCE LENGTH DISTRIBUTION:
  1: 1 workload(s)
  7: 1 workload(s)
  14: 1 workload(s)
  15: 1 workload(s)
  16: 1 workload(s)
  32: 1 workload(s)
  52: 1 workload(s)
  53: 1 workload(s)
  54: 1 workload(s)
  55: 1 workload(s)
  56: 1 workload(s)
  57: 1 workload(s)
  58: 1 workload(s)
  59: 1 workload(s)
  62: 1 workload(s)
  80: 1 workload(s)
  901: 1 workload(s)
  11948: 1 workload(s)
  14107: 1 workload(s)

OPTIMIZATION INSIGHTS:
1. BATCHING OPPORTUNITIES:
   - Multiple workloads with same seq_len can be batched
   - Consider dynamic batching for efficiency

2. MEMORY ACCESS PATTERNS:
   - Routing logits: [seq_len, 256] (row-major)
   - Routing bias: [256] (vector)
   - Bias broadcast needed across sequence dimension

3. COMPUTE PATTERNS:
   - Top-8 selection per sequence element
   - Expert indices: 0-255
   - Expert offset handling: 0-224

4. KERNEL DESIGN RECOMMENDATIONS:
   - Handle variable seq_len with parameterized kernels
   - Opt

In [12]:
# Summary Table for Kernel Development
import pandas as pd

print("=== KERNEL DEVELOPMENT SUMMARY TABLE ===")
print()

summary_data = []
for analysis in tensor_analysis:
    summary_data.append({
        'Workload_ID': analysis['workload_id'],
        'Seq_Len': analysis['seq_len'],
        'Expert_Offset': analysis['expert_offset'],
        'Logits_Shape': f"{analysis['logits_shape'][0]}x{analysis['logits_shape'][1]}",
        'Bias_Shape': f"{analysis['bias_shape'][0]}",
        'Memory_MB': round((analysis['seq_len'] * analysis['logits_shape'][1] * 2) / (1024 * 1024), 3)
    })

df = pd.DataFrame(summary_data)
print(df.to_string(index=False))

print(f"\n=== KERNEL CONSTANTS ===")
print(f"NUM_EXPERTS: {tensor_analysis[0]['logits_shape'][1]}")
print(f"TOP_K: 8")
print(f"NUM_GROUPS: 8")
print(f"EXPERTS_PER_GROUP: 4")
print(f"DATA_TYPE: {sample_logits.dtype}")
print(f"SCALING_FACTOR: 2.5")

print(f"\n=== VARIABLE PARAMETERS ===")
print(f"SEQ_LEN: {min(seq_lengths)} to {max(seq_lengths)}")
print(f"EXPERT_OFFSET: {min(expert_offsets)} to {max(expert_offsets)}")

print(f"\n=== MEMORY BUDGET ===")
print(f"Max routing logits: {(max(seq_lengths) * tensor_analysis[0]['logits_shape'][1] * 2) / (1024 * 1024):.3f} MB")
print(f"Routing bias: {(tensor_analysis[0]['bias_shape'][0] * 2) / (1024 * 1024):.3f} MB")
print(f"Total max: {((max(seq_lengths) * tensor_analysis[0]['logits_shape'][1] + tensor_analysis[0]['bias_shape'][0]) * 2) / (1024 * 1024):.3f} MB")

=== KERNEL DEVELOPMENT SUMMARY TABLE ===

 Workload_ID  Seq_Len  Expert_Offset Logits_Shape Bias_Shape  Memory_MB
           1        7            192        7x256        256      0.003
           2        1             32        1x256        256      0.000
           3       32             32       32x256        256      0.016
           4       80             96       80x256        256      0.039
           5      901             96      901x256        256      0.440
           6       16            224       16x256        256      0.008
           7       15             32       15x256        256      0.007
           8       14              0       14x256        256      0.007
           9    14107             32    14107x256        256      6.888
          10    11948            128    11948x256        256      5.834
          11       62             96       62x256        256      0.030
          12       59            160       59x256        256      0.029
          13       58 

In [2]:
# VRAM Usage Analysis for Different Sequence Lengths
print("=== VRAM USAGE ANALYSIS ===")
print()

# Fixed parameters from DeepSeek-V3 MoE
H = 7168  # hidden_size
I = 2048  # intermediate_size  
E_global = 256  # global experts
E_local = 32  # local experts
BLOCK = 128

# Test different sequence lengths
test_seq_lengths = [1, 32, 512, 2048, 8192, 14107]

def bytes_to_gb(bytes_val):
    return bytes_val / (1024**3)

print("Sequence Length | Input VRAM | Weights VRAM | Scales VRAM | Total VRAM")
print("-" * 70)

for seq_len in test_seq_lengths:
    # Input tensors
    routing_logits = seq_len * E_global * 4  # float32
    routing_bias = E_global * 2  # bfloat16
    hidden_states = seq_len * H * 2  # bfloat16
    hidden_states_scale = (H // BLOCK) * seq_len * 4  # float32
    
    # Weight tensors (FP8 quantized)
    gemm1_weights = E_local * (2 * I) * H * 1  # fp8
    gemm2_weights = E_local * H * I * 1  # fp8
    
    # Scale tensors
    gemm1_scales = E_local * ((2 * I) // BLOCK) * (H // BLOCK) * 4  # float32
    gemm2_scales = E_local * (H // BLOCK) * (I // BLOCK) * 4  # float32
    
    # Intermediate tensors (during computation)
    dequant_hidden = seq_len * H * 4  # float32
    dequant_gemm1 = E_local * (2 * I) * H * 4  # float32
    dequant_gemm2 = E_local * H * I * 4  # float32
    output = seq_len * H * 2  # bfloat16
    
    # Calculate totals
    input_vram = routing_logits + routing_bias + hidden_states + hidden_states_scale
    weights_vram = gemm1_weights + gemm2_weights
    scales_vram = gemm1_scales + gemm2_scales
    total_vram = input_vram + weights_vram + scales_vram + dequant_hidden + dequant_gemm1 + dequant_gemm2 + output
    
    print(f"{seq_len:13d} | {bytes_to_gb(input_vram):9.3f}GB | {bytes_to_gb(weights_vram):10.3f}GB | {bytes_to_gb(scales_vram):9.3f}GB | {bytes_to_gb(total_vram):10.3f}GB")

print(f"\n=== VRAM BREAKDOWN FOR MAX SEQ_LEN ({max(test_seq_lengths)}) ===")
print(f"Routing Logits: {bytes_to_gb(routing_logits):.3f} GB")
print(f"Routing Bias: {bytes_to_gb(routing_bias):.3f} GB") 
print(f"Hidden States: {bytes_to_gb(hidden_states):.3f} GB")
print(f"Hidden States Scale: {bytes_to_gb(hidden_states_scale):.3f} GB")
print(f"GEMM1 Weights (FP8): {bytes_to_gb(gemm1_weights):.3f} GB")
print(f"GEMM2 Weights (FP8): {bytes_to_gb(gemm2_weights):.3f} GB")
print(f"GEMM1 Scales: {bytes_to_gb(gemm1_scales):.3f} GB")
print(f"GEMM2 Scales: {bytes_to_gb(gemm2_scales):.3f} GB")
print(f"Dequantized Tensors: {bytes_to_gb(dequant_hidden + dequant_gemm1 + dequant_gemm2):.3f} GB")
print(f"Output: {bytes_to_gb(output):.3f} GB")
print(f"TOTAL: {bytes_to_gb(total_vram):.3f} GB")

=== VRAM USAGE ANALYSIS ===

Sequence Length | Input VRAM | Weights VRAM | Scales VRAM | Total VRAM
----------------------------------------------------------------------
            1 |     0.000GB |      1.312GB |     0.000GB |      6.563GB
           32 |     0.000GB |      1.312GB |     0.000GB |      6.565GB
          512 |     0.007GB |      1.312GB |     0.000GB |      6.591GB
         2048 |     0.030GB |      1.312GB |     0.000GB |      6.675GB
         8192 |     0.119GB |      1.312GB |     0.000GB |      7.010GB
        14107 |     0.205GB |      1.312GB |     0.000GB |      7.333GB

=== VRAM BREAKDOWN FOR MAX SEQ_LEN (14107) ===
Routing Logits: 0.013 GB
Routing Bias: 0.000 GB
Hidden States: 0.188 GB
Hidden States Scale: 0.003 GB
GEMM1 Weights (FP8): 0.875 GB
GEMM2 Weights (FP8): 0.438 GB
GEMM1 Scales: 0.000 GB
GEMM2 Scales: 0.000 GB
Dequantized Tensors: 5.627 GB
Output: 0.188 GB
TOTAL: 7.333 GB


In [1]:
import urllib.request
import json
# TODO: Replace with your actual roll number
ROLL_NUMBER = 2025701040
INDEX = 0
url = f"http://preon.iiit.ac.in:8026/api/data?roll={ROLL_NUMBER}&index={INDEX}"
try:
    with urllib.request.urlopen(url) as response:
        if response.status == 200:
            data = json.loads(response.read().decode())
            print(f"Features: {data['features']}")
            print(f"Label: {data['label']}")
except urllib.error.HTTPError as e:
    if e.code == 404:
        print("End of dataset reached. Stop collecting.")
    else:
        print(f"An HTTP error occurred: {e.code}")

Features: [1.442314693770793, 7.144374835963619, 3.639369832197773, 0.5961967498221038, 1.200659706286487]
Label: 0
